# Phase 4 · Попередній експеримент: дистиляція GPT-2 Medium → DistilGPT-2

**Роль у LiteHorizon:** окремий методологічний орієнтир для **Phase 4 (адаптація моделей)**. Досліджується дистиляція *розподілів токенів* (logit/soft-target distillation), а не генерація розмітки Gemini та не LoRA. Цей код **не** запускає `horizon` і не відтворює E4 статті IT&I 2026.

**Протокол:** нерухомий учитель `gpt2-medium` · студент `distilgpt2` · WikiText-2 (`Salesforce/wikitext`, `wikitext-2-raw-v1`) · 2000 навчальних / 500 валідаційних записів · 2 епохи · batch size 8 · AdamW (`lr=5e-5`) · 128 токенів · ціль `0.7 × CE + 0.3 × T² × KL`, `T=2`, з маскуванням паддингу.

Виконайте комірки згори донизу. Інструкція зі встановлення, застереження та зв'язок із основним проєктом наведені в сусідньому **`README.md`**. У кінці є **архівні результати**, перенесені з вихідного ноутбука; вони не є результатами поточного запуску.

## 1. Імпорт залежностей

In [ ]:
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
import math
from tqdm import tqdm

## 2. Ініціалізація моделей і токенізатора

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
teacher = AutoModelForCausalLM.from_pretrained("gpt2-medium").to(device)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False
student = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
t_params = sum(p.numel() for p in teacher.parameters())
s_params = sum(p.numel() for p in student.parameters())
print(f"Teacher: {t_params:,} params")
print(f"Student: {s_params:,} params")
print(f"Compression: {t_params / s_params:.1f}x")

## 3. Функція втрат

In [ ]:
def distillation_loss(
    student_logits,
    teacher_logits,
    input_ids,
    attention_mask,
    temperature=2.0,
    alpha=0.3,
):
    student_logits = student_logits[:, :-1, :].contiguous()
    teacher_logits = teacher_logits[:, :-1, :].contiguous()

    labels = input_ids[:, 1:].contiguous()
    mask = attention_mask[:, 1:].contiguous()

    ce_loss = F.cross_entropy(
        student_logits.view(-1, student_logits.size(-1)),
        labels.view(-1),
        reduction="none",
    )

    ce_loss = ce_loss.view_as(labels)
    ce_loss = (ce_loss * mask).sum() / mask.sum().clamp_min(1)

    student_log_probs = F.log_softmax(
        student_logits / temperature,
        dim=-1,
    )

    # Teacher is a fixed target.
    teacher_probs = F.softmax(
        teacher_logits.detach() / temperature,
        dim=-1,
    )

    kd_loss = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="none",
    ).sum(dim=-1)


    kd_loss = (kd_loss * mask).sum() / mask.sum().clamp_min(1)
    kd_loss = kd_loss * (temperature ** 2)

    loss = (1.0 - alpha) * ce_loss + alpha * kd_loss

    return loss

## 4. Навчання на WikiText-2

In [ ]:
dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="train"
)
dataset = dataset.filter(lambda x: len(x['text'].strip()) > 50)

def tokenize(examples):
    return tokenizer(examples['text'], truncation=True,
        max_length=128, padding='max_length')

tokenized = dataset.select(range(2000)).map(
    tokenize, batched=True, remove_columns=['text'])
tokenized.set_format('torch')

loader = DataLoader(tokenized, batch_size=8, shuffle=True)
optimizer = torch.optim.AdamW(student.parameters(), lr=5e-5)
student.train()

for epoch in range(2):
    total_loss = 0.0

    for batch in tqdm(loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)

        optimizer.zero_grad()
        student_out = student(
            input_ids,
            attention_mask=attn
        )
        with torch.no_grad():
            teacher_out = teacher(
                input_ids,
                attention_mask=attn
            )
        loss = distillation_loss(
            student_out.logits,
            teacher_out.logits,
            input_ids,
            attn
        )
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} avg loss: {total_loss / len(loader):.4f}")

## 5. Валідаційна perplexity

In [ ]:
def compute_perplexity(model, eval_loader):
    model.eval()
    total_nll = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in eval_loader:
            ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)

            labels = ids.clone()

            # Ignore padding tokens in cross-entropy
            labels[attn == 0] = -100

            out = model(
                input_ids=ids,
                attention_mask=attn,
                labels=labels
            )

            num_tokens = (labels[:, 1:] != -100).sum().item()

            total_nll += out.loss.item() * num_tokens
            total_tokens += num_tokens

    return math.exp(total_nll / total_tokens)
            
eval_data = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="validation")
eval_data = eval_data.filter(lambda x: len(x['text'].strip()) > 50)
eval_tok = eval_data.select(range(500)).map(
    tokenize, batched=True, remove_columns=['text'])
eval_tok.set_format('torch')
eval_loader = DataLoader(eval_tok, batch_size=8)
original = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)

print(f"Teacher perplexity: {compute_perplexity(teacher, eval_loader):.2f}")
print(f"Original student perplexity: {compute_perplexity(original, eval_loader):.2f}")
print(f"Distilled student perplexity: {compute_perplexity(student, eval_loader):.2f}")

## Архівні результати оригінального запуску

| Показник | Значення |
|---|---:|
| Параметрів учителя | 354 823 168 |
| Параметрів студента | 81 912 576 |
| Відношення параметрів учителя до студента | ≈4,3 |
| Середня навчальна втрата, епоха 1 | 3,1024 |
| Середня навчальна втрата, епоха 2 | 2,8487 |
| Тривалість епохи 1 у збереженому журналі | 28 хв 42 с |
| Тривалість епохи 2 у збереженому журналі | 28 хв 57 с |
| Perplexity учителя | 36,75 |
| Perplexity початкового студента | 74,28 |
| **Perplexity студента після дистиляції** | **38,97** |

*Ці числа витягнуто зі збережених виводів старого ноутбука. Навчання не повторювалося під час очищення. `Random seed` не було зафіксовано; порівняння 38,97 з 43,32 є лише спостереженням, не контрольованим експериментом. Навчальні втрати CE та CE+KL не можна безпосередньо порівнювати. Ці perplexity не є показниками якості сканування горизонту.*